# Diário 1 — explorar o bruto e acompanhar a limpeza

Começo pelo CSV original do CORD-19, release de 02/06/2022. Quero identificar o que está disponível e acompanhar o efeito de cada critério antes de discutir o enriquecimento. As análises usam todos os registros; as poucas linhas exibidas são exemplos, sem sorteio estatístico.

## Preparar a leitura

As configurações ficam em `src/config.py`. A função `clean_frame()` concentra a regra compartilhada de limpeza. As contagens e inspeções exploratórias permanecem nas células. Este notebook trabalha em memória e não grava os Parquet.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display


root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root))

from src import config as cfg

pd.set_option("display.max_colwidth", 120)
from src.io_utils import atomic_json, atomic_parquet
cfg.ensure_directories()


## Abrir a base original

`metadata_path()` exige um arquivo local: não inicia download. Coloque o CSV em `data/raw/cord19_2022-06-02/metadata.csv` ou informe `CORD19_METADATA`. A leitura mantém as mesmas seis colunas, tipos e marcadores de ausência da leitura atual.

In [2]:
caminho_bruto = cfg.metadata_path()
bruto = pd.read_csv(caminho_bruto, usecols=cfg.COLUMNS, dtype=str,
                    na_values=cfg.NULL_VALUES)[cfg.COLUMNS]
display(Markdown(f"**Fonte:** `{caminho_bruto}` · **Registros lidos:** {len(bruto):,}"))
display(bruto.head(5))
display(bruto.dtypes.rename("Tipo lido").to_frame())

**Fonte:** `C:\UTFPR\TCC\tcc-data-viz\tcc-data-viz-main\data\raw\cord19_2022-06-02\metadata.csv` · **Registros lidos:** 1,056,660

,cord_uid,doi,title,abstract,authors,publish_time
0,ug7v899j,10.1186/1471-2334-1-6,"Clinical features of culture-proven Mycoplasma pneumoniae infections at King Abdulaziz University Hospital, Jeddah, ...",OBJECTIVE: This retrospective chart review describes the epidemiology and clinical features of 40 patients with cult...,"Madani, Tariq A; Al-Ghamdi, Aisha A",2001-07-04
1,02tnwd4m,10.1186/rr14,Nitric oxide: a pro-inflammatory mediator in lung disease?,Inflammatory diseases of the respiratory tract are commonly associated with elevated production of nitric oxide (NO•...,"Vliet, Albert van der; Eiserich, Jason P; Cross, Carroll E",2000-08-15
2,ejv2xln0,10.1186/rr19,Surfactant protein-D and pulmonary host defense,"Surfactant protein-D (SP-D) participates in the innate response to inhaled microorganisms and organic antigens, and ...","Crouch, Erika C",2000-08-25
3,2b73a28n,10.1186/rr44,Role of endothelin-1 in lung disease,Endothelin-1 (ET-1) is a 21 amino acid peptide with diverse biological activity that has been implicated in numerous...,"Fagan, Karen A; McMurtry, Ivan F; Rodman, David M",2001-02-22
4,9785vg6d,10.1186/rr61,Gene expression in epithelial cells in response to pneumovirus infection,"Respiratory syncytial virus (RSV) and pneumonia virus of mice (PVM) are viruses of the family Paramyxoviridae, subfa...","Domachowske, Joseph B; Bonville, Cynthia A; Rosenberg, Helene F",2001-05-11


,Tipo lido
cord_uid,str
doi,str
title,str
abstract,str
authors,str
publish_time,str


## Quais campos estão ausentes?

Calculo ausências e percentuais no bruto inteiro. Um mesmo artigo pode aparecer em várias linhas desta tabela: as ausências por campo não são grupos independentes. O resumo é necessário para o LDA, e autores/datas são necessários para a análise de publicações.

In [3]:
ausencias = bruto.isna().sum().to_frame("Ausências")
ausencias["Percentual (%)"] = (100 * ausencias["Ausências"] / len(bruto)).round(2)
display(ausencias)

,Ausências,Percentual (%)
cord_uid,0,0.00
doi,399880,37.84
title,503,0.05
abstract,235544,22.29
authors,23869,2.26
publish_time,1814,0.17


## Identificadores repetidos e DOI ausente

Inspeciono os registros sem identificador e as ocorrências repetidas que seriam descartadas. A regra mantém a primeira ocorrência de cada `cord_uid` e só depois exige DOI. Por isso, contar DOIs ausentes no bruto não substitui medir esse filtro após a deduplicação.

In [4]:
id_presente = bruto["cord_uid"].notna()
repetidos = id_presente & bruto["cord_uid"].duplicated(keep="first")
diagnostico = pd.Series({
    "Sem cord_uid": int((~id_presente).sum()),
    "Ocorrências repetidas descartáveis": int(repetidos.sum()),
    "DOI ausente no bruto": int(bruto["doi"].isna().sum()),
}, name="Registros")
display(diagnostico.to_frame())
display(bruto.loc[repetidos, ["cord_uid", "doi", "title"]].head(5))
display(bruto.loc[bruto["doi"].isna(), ["cord_uid", "doi", "title"]].head(5))

,Registros
Sem cord_uid,0
Ocorrências repetidas descartáveis,85824
DOI ausente no bruto,399880


,cord_uid,doi,title
13319,esnam1e1,10.23750/abm.v91i2.9514,The death rate for COVID-19 is positively associated with gross domestic products
15859,j3755ixl,10.23750/abm.v91i4.10698,An Update on Pulmonary Hypertension in Coronavirus Disease-19 (COVID-19)
51864,xdktx9n2,10.1007/s11064-017-2390-x,Neural Progenitor Cells Undergoing Yap/Tead-Mediated Enhanced Self-Renewal Form Heterotopias More Easily in the Dien...
53813,d0tsfrm2,10.1016/s2666-0849(20)31119-0,Full Issue PDF
54067,j3d4w22c,10.1016/j.ajpath.2020.12.003,This Month in AJP


,cord_uid,doi,title
282393,f3hoz9bu,NaN,"Recently published papers: inflammation, elucidation, manipulation?"
282394,29mhdw6y,NaN,An investigation of the role of transmembrane domains in Golgi protein retention.
282395,jblw7kpf,NaN,Unique features of internal initiation of hepatitis C virus RNA translation.
282396,ayydekzc,NaN,Structural and functional studies of retroviral RNA pseudoknots involved in ribosomal frameshifting: nucleotides at ...
282397,dbstq3x1,NaN,Topology of the non-structural rotavirus receptor glycoprotein NS28 in the rough endoplasmic reticulum.


## Aplicar os critérios de limpeza
Removo identificadores ausentes, mantenho a primeira ocorrência de cada `cord_uid` e exijo DOI.
Em seguida, separo os registros completos dos que precisam de enriquecimento.

In [5]:
ids_unicos = bruto.dropna(subset=["cord_uid"]).drop_duplicates("cord_uid", keep="first")
com_doi = ids_unicos.dropna(subset=["doi"])
mascara_completos = com_doi[cfg.COLUMNS].notna().all(axis=1)
completos = com_doi.loc[mascara_completos].reset_index(drop=True)
incompletos = com_doi.loc[~mascara_completos].reset_index(drop=True)
contagens = {"bruto": len(bruto), "cord_uid_unicos": len(ids_unicos), "com_doi": len(com_doi),
             "completas": len(completos), "incompletas": len(incompletos)}
display(pd.Series(contagens, name="Artigos").to_frame())

,Artigos
bruto,1056660
cord_uid_unicos,970836
com_doi,654783
completas,529470
incompletas,125313


## Examinar o grupo que será enriquecido

O diagnóstico abaixo usa somente os registros incompletos com DOI. Esse é o denominador da comparação antes/depois do próximo notebook. Mostrar exemplos com ausências ajuda a compreender o problema que o checkpoint tenta resolver.

In [6]:
campos_enriquecidos = cfg.COLUMNS[2:]
pendencias = incompletos[campos_enriquecidos].isna().sum().to_frame("Ausências antes")
pendencias["Percentual no grupo (%)"] = (100 * pendencias["Ausências antes"] / len(incompletos)).round(2)
display(pendencias)
display(incompletos.sort_values("cord_uid").head(5))

,Ausências antes,Percentual no grupo (%)
title,495,0.40
abstract,121512,96.97
authors,17211,13.73
publish_time,1085,0.87


,cord_uid,doi,title,abstract,authors,publish_time
27498,00012cqv,10.1001/jamanetworkopen.2021.13941,Peer-to-Peer Support for Older Adults-What Do We Know and Where Do We Go?,NaN,"Thombs, Brett D; Carboni-Jiménez, Andrea",2021-06-01
121866,0007wzmf,10.1007/s00404-022-06422-6,Trends in the risk of second primary malignancies among survivors of ovarian clear cell carcinoma,NaN,"Wang, Yiru; Guo, Ying; Li, Haiyan",2022
102045,000xcuss,10.31744/einstein_journal/2021ce6419,Use of mouthwashes in the management of COVID-19 patients in intensive care units: recommendations and current evidence,NaN,"Tovani-Palone, Marcos Roberto; Shamsoddin, Erfan",2021-03-17
92778,0011irx9,10.1016/s2213-2600(21)00123-5,Young person with long COVID after mild disease,NaN,"Kirby, Tony",2021-05-31
1848,001wbz6e,10.1016/0958-1669(95)80119-7,Pharmaceutical biotechnology,NaN,NaN,2002-02-25


## Gravar os dois grupos da execução atual
Estes Parquet serão as entradas do notebook 02. As contagens ficam em JSON para conferência.

In [7]:
atomic_parquet(cfg.STAGED / "01_completas.parquet", completos)
atomic_parquet(cfg.STAGED / "01_incompletas.parquet", incompletos)
atomic_json(cfg.OUTPUTS / "tables/preparacao.json", contagens)
display(Markdown("A limpeza atual foi salva. O próximo notebook recupera os campos ausentes."))

A limpeza atual foi salva. O próximo notebook recupera os campos ausentes.

## Fechamento
A exigência de DOI e de metadados completos restringe o universo analisado.
Continuo a curadoria nos mesmos registros, utilizando as respostas bibliográficas fornecidas.